# RAG Pipeline for Scientific Papers
## Notebook 02 — Chunking, Embedding & Retrieval

**Reads:** `output/text/page_*.txt` (from notebook 01)  
**Writes:** `output/chroma_db/` (persistent vector store)

```
page text files
  → split into overlapping chunks
    → embed with sentence-transformers (local, free)
      → store in ChromaDB
        → query → top-3 most relevant chunks
```

## 0. Install & Imports

In [1]:
# Install if needed (run once)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "sentence-transformers", "chromadb", "-q"], check=True)
print("Dependencies ready")

Dependencies ready


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaqpotpy 7.1.0 requires pandas>=2.2.2, but you have pandas 2.1.4 which is incompatible.
jaqpotpy 7.1.0 requires scikit-learn>=1.5.0, but you have scikit-learn 1.4.2 which is incompatible.


In [2]:
import json
import re
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

# ── Paths (same root as notebook 01) ─────────────────────────────────────────
ROOT_DIR   = Path.cwd().parent
OUTPUT_DIR = ROOT_DIR / "output"
TEXT_DIR   = OUTPUT_DIR / "text"
CHROMA_DIR = OUTPUT_DIR / "chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

# ── Chunking settings ─────────────────────────────────────────────────────────
CHUNK_SIZE    = 400   # characters per chunk
CHUNK_OVERLAP = 80    # overlap between consecutive chunks

# ── Embedding model ───────────────────────────────────────────────────────────
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # fast, 384-dim, good quality
COLLECTION_NAME = "paper_chunks"
TOP_K           = 3

print(f"ROOT       : {ROOT_DIR}")
print(f"TEXT_DIR   : {TEXT_DIR}")
print(f"CHROMA_DIR : {CHROMA_DIR}")
print(f"Text files : {len(list(TEXT_DIR.glob('*.txt')))}")

ROOT       : /Users/alexandrosangelis/Documents/VS_CODE/rag-paper-pipeline
TEXT_DIR   : /Users/alexandrosangelis/Documents/VS_CODE/rag-paper-pipeline/output/text
CHROMA_DIR : /Users/alexandrosangelis/Documents/VS_CODE/rag-paper-pipeline/output/chroma_db
Text files : 13


## 1. Load Extracted Text
Read all `page_*.txt` files produced by notebook 01.

In [3]:
def load_pages(text_dir: Path) -> list[dict]:
    """Load all page text files, sorted by page number."""
    pages = []
    for txt_file in sorted(text_dir.glob("page_*.txt")):
        page_num = int(txt_file.stem.split("_")[1])
        text     = txt_file.read_text(encoding="utf-8").strip()
        if text:   # skip empty pages
            pages.append({"page": page_num, "text": text})
    return pages


pages = load_pages(TEXT_DIR)
print(f"Loaded {len(pages)} pages")
print(f"Total chars: {sum(len(p['text']) for p in pages):,}")

Loaded 13 pages
Total chars: 50,293


## 2. Chunking

Split each page into overlapping chunks.

**Why overlap?** A sentence at the boundary of a chunk would lose its context without it. With overlap, each chunk shares some text with the next, so retrieved chunks always contain complete thoughts.

```
page text:  [----chunk 1----]
                        [----chunk 2----]
                                    [----chunk 3----]
                         ↑ overlap  ↑ overlap
```

In [4]:
def split_into_chunks(text: str, chunk_size: int, overlap: int) -> list[str]:
    """
    Split text into overlapping chunks.
    Tries to break at sentence boundaries ('. ') when possible.
    """
    chunks = []
    start  = 0

    while start < len(text):
        end = start + chunk_size

        if end < len(text):
            # Try to break at a sentence boundary near the end
            boundary = text.rfind('. ', start, end)
            if boundary != -1 and boundary > start + chunk_size // 2:
                end = boundary + 1   # include the period

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        start = end - overlap

    return chunks


def build_chunks(pages: list[dict], chunk_size: int, overlap: int) -> list[dict]:
    """Chunk all pages. Each chunk carries page number and position metadata."""
    all_chunks = []
    for page in pages:
        chunks = split_into_chunks(page["text"], chunk_size, overlap)
        for i, chunk_text in enumerate(chunks):
            all_chunks.append({
                "id"       : f"p{page['page']:03d}_c{i:03d}",
                "text"     : chunk_text,
                "page"     : page["page"],
                "chunk_idx": i,
            })
    return all_chunks


chunks = build_chunks(pages, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"Total chunks : {len(chunks)}")
print(f"Avg chunk len: {sum(len(c['text']) for c in chunks) / len(chunks):.0f} chars")
print()
print("── Sample chunk (page 1, chunk 0) ──")
print(chunks[0]['text'])
print()
print("── Sample chunk (page 4, chunk 0) ──")
p4 = [c for c in chunks if c['page'] == 4]
if p4: print(p4[0]['text'])

Total chunks : 187
Avg chunk len: 342 chars

── Sample chunk (page 1, chunk 0) ──
International Journal o
Open Access Full Text Article
Endocytosis an
in mammalian
Nuri Oh1,2
Ji-Ho Park1–3
1Department of Bio and Brain
Engineering, 2Institute for Optical
Science and Technology, 3Institute for
the NanoCentury, Korea Advanced
Institute of Science and Technology
(KAIST), Daejeon, Republic of Korea
Correspondence: Ji-Ho Park
Department of Bio and Brain
Engineering, Institute for Opt

── Sample chunk (page 4, chunk 0) ──
Oh and Park
endocytosis indicates receptor-mediated endocytosis. Many
types of cells use the clathrin- and caveolae-mediated
endocytosis pathways to internalize nanoscale materials,
including viruses and nanoparticles.19–21 These endocytosis
pathways are the most important pathways for the internalization of nanoparticles into cells because the nanoparticles
are directly coated with the plasma pro


## 3. Embed & Store in ChromaDB

`all-MiniLM-L6-v2` converts each chunk into a 384-dimensional vector.  
ChromaDB stores vectors + text + metadata on disk in `output/chroma_db/`.

In [5]:
# Load embedding model (downloads ~90 MB on first run)
print(f"Loading embedding model '{EMBEDDING_MODEL}'...")
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"✅ Model loaded — embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading embedding model 'all-MiniLM-L6-v2'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded — embedding dimension: 384


/var/folders/11/3c5hy62x5sbcbgmx_l91krkr0000gn/T/ipykernel_10478/1009152736.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model loaded — embedding dimension: {model.get_sentence_embedding_dimension()}")


In [6]:
# Set up ChromaDB (persistent on disk)
client     = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Delete existing collection if re-running (fresh start)
try:
    client.delete_collection(COLLECTION_NAME)
    print("Existing collection cleared")
except:
    pass

collection = client.create_collection(
    name     = COLLECTION_NAME,
    metadata = {"hnsw:space": "cosine"}   # cosine similarity
)

# Embed all chunks
print(f"Embedding {len(chunks)} chunks...")
texts      = [c["text"] for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

# Store in ChromaDB
collection.add(
    ids        = [c["id"] for c in chunks],
    embeddings = embeddings.tolist(),
    documents  = texts,
    metadatas  = [{"page": c["page"], "chunk_idx": c["chunk_idx"]} for c in chunks],
)

print(f"\n✅ {collection.count()} chunks stored in ChromaDB")
print(f"   Persisted at: {CHROMA_DIR}")

Embedding 187 chunks...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


✅ 187 chunks stored in ChromaDB
   Persisted at: /Users/alexandrosangelis/Documents/VS_CODE/rag-paper-pipeline/output/chroma_db


## 4. Retrieval — Ask a Question

The query is embedded with the same model, then ChromaDB finds the top-k chunks with the highest cosine similarity.

In [7]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Embed query and return top_k most relevant chunks."""
    query_embedding = model.encode([query])[0].tolist()

    results = collection.query(
        query_embeddings = [query_embedding],
        n_results        = top_k,
        include          = ["documents", "metadatas", "distances"],
    )

    hits = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        hits.append({
            "text"       : doc,
            "page"       : meta["page"],
            "chunk_idx"  : meta["chunk_idx"],
            "similarity" : round(1 - dist, 4),   # cosine distance → similarity
        })
    return hits


def print_results(query: str, hits: list[dict]):
    print(f"Query: '{query}'")
    print("═" * 60)
    for i, hit in enumerate(hits, 1):
        print(f"\n#{i} | Page {hit['page']} | Similarity: {hit['similarity']:.4f}")
        print("─" * 60)
        print(hit['text'])
    print("\n" + "═" * 60)

In [9]:
# ── Try a query ───────────────────────────────────────────────────────────────
QUERY = "what is endocytosis?"

hits = retrieve(QUERY, top_k=TOP_K)
print_results(QUERY, hits)

Query: 'what is endocytosis?'
════════════════════════════════════════════════════════════

#1 | Page 3 | Similarity: 0.6627
────────────────────────────────────────────────────────────
ignaling molecules to obtain energy
and interact with other cells, respectively. The endocytosis
pathways are typically classified into clathrin- and caveolaemediated endocytosis, phagocytosis, macropinocytosis,
and pinocytosis (Table 1).

#2 | Page 3 | Similarity: 0.6455
────────────────────────────────────────────────────────────
s of cells in the body use the endocytosis process to
communicate with the biological environments. This process
is an energy-dependent process through which cells internalize ions and biomolecules.18 In particular, the cells internalize nutrients and signaling molecules to obtain energy
and interact with other cells, respectively.

#3 | Page 13 | Similarity: 0.6050
────────────────────────────────────────────────────────────
pl 1)

Endocytosis and exocytosis of nanoparticles

In [10]:
# ── Try more queries ──────────────────────────────────────────────────────────
queries = [
    "What is the role of surface chemistry in nanoparticle uptake?",
    "How do macrophages respond to nanoparticles?",
    "What are the exocytosis mechanisms for nanoparticles?",
]

for q in queries:
    hits = retrieve(q, top_k=TOP_K)
    print_results(q, hits)
    print()

Query: 'What is the role of surface chemistry in nanoparticle uptake?'
════════════════════════════════════════════════════════════

#1 | Page 6 | Similarity: 0.7248
────────────────────────────────────────────────────────────
Oh and Park
A B
50 nm
C
12
Size (nm)
50 nm
Surface chemistry
Surface chemistry (or surface charge) of nanoparticles can
be determined by the chemical composition on the nanoparticle surface. The surface charge of nanoparticles can affect
their efficiency and the pathway of cellular uptake, because
biological systems consist of numerous biomolecules with
various charges.

#2 | Page 4 | Similarity: 0.6745
────────────────────────────────────────────────────────────
nanoparticle size has been known to be a key determinant of
the uptake pathways. Many critical in vivo functions of nanoparticles, such as circulation time, targeting, internalization,
and clearance, depend on their size.

#3 | Page 2 | Similarity: 0.6315
─────────────────────────────────────────────────

## 5. Interactive Query
Change the query below and re-run to explore the paper.

In [11]:
# ── Your custom query ─────────────────────────────────────────────────────────
MY_QUERY = "cytotoxicity of nanoparticles"   # ← change this

hits = retrieve(MY_QUERY, top_k=TOP_K)
print_results(MY_QUERY, hits)

Query: 'cytotoxicity of nanoparticles'
════════════════════════════════════════════════════════════

#1 | Page 1 | Similarity: 0.6985
────────────────────────────────────────────────────────────
ionalized nanoparticles based on various sizes, shapes, and surface chemistries. We believe
that this review contributes to the design of safe nanoparticles that can efficiently enter and
leave human cells and tissues.
Keywords: drug delivery, endocytosis, exocytosis, cancer cell, macrophage, nanoparticle,
toxicity
Introduction
Nano-sized materials have been increasingly used in the medical field

#2 | Page 1 | Similarity: 0.6778
────────────────────────────────────────────────────────────
hown some limitations regarding
the toxicity of the nanoscale materials in the body.4,5 In order to reduce their toxicity, it
is crucial to study endocytosis, exocytosis, and clearance mechanisms for nanoparticles
released from the nanoparticle–drug conjugates. Nanoparticles exposed to the bloodstream interac

---
## Summary

| Step | What happened |
|---|---|
| Chunking | Pages split into ~400-char overlapping chunks |
| Embedding | Each chunk → 384-dim vector via `all-MiniLM-L6-v2` |
| Storage | Vectors + text persisted in ChromaDB on disk |
| Retrieval | Query embedded → cosine similarity → top-3 chunks |

**Next step:** pass the retrieved chunks + the original query to a language model (Claude/GPT) to generate a grounded answer — that's the full RAG loop.